# Soft-SPIN Healthcare — Step 2: HSIC (decoupling) + capability, swept over α

Loads the Soft Step-1 coupled set (`coupled_soft_health.json`), loads the model once, caches the original
coupled-weight values once, then sweeps the α grid **internally**: for each α it resets the coupled weights
to `α·W` (absolute assignment — soft scaling is not idempotent) and measures the **full decoupling pack** +
capability. α=0 reproduces hard SPIN; α=1 reproduces baseline. Forward passes only — no generation, no judge.

**Full stat pack (matches the general/iter health Step 2):** raw HSIC (final + per-layer, σ∈{50,100,200,400})
+ **normalised HSIC** + **linear CKA** + **representation-norm diagnostics**, paired bootstrap 95% CIs on the
HSIC drop, and capability = WikiText-2 + PubMedQA perplexity, MedMCQA + MMLU-clinical accuracy. Per-subset
HSIC is omitted from the sweep (extra passes per α) — measure once at the chosen α.

**Prediction under test:** HSIC / nHSIC / CKA rise monotonically with α (0 → baseline); perplexity best near
α=1, worst near α=0. α=0 should reproduce the hard-SPIN Step-2 numbers (ΔHSIC@400 ≈ −5.2%); α=1 ≈ baseline.

**Inputs (Kaggle Dataset inputs):** `coupled_soft_health.json` (Soft Step-1), `eval_fair_health.csv`,
`eval_priv_health.csv` (general Step-2 splits), `cal_general_health.csv` (to exclude PubMedQA calibration from
the ppl corpus). Internet ON. GPU **T4×2**. ~1.5–2 h. **Output:** `metrics_soft_health.json` (checkpointed
per α → resumable; the bootstrap needs a single clean session, since reps are held in memory only).

## Step 0 — config

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, re, gc, time, json
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0','sentencepiece'], check=False)
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

MODEL_ID='Qwen/Qwen2-7B-Instruct'
USE_FAST=False
ALPHAS=[0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]   # 0 = hard SPIN, 1 = baseline
HSIC_N=300; PPL_SEQLEN=2048; SIGMAS=[50,100,200,400]
USE_CHAT_TEMPLATE=False; DTYPE=torch.float16; BOOT=500; SEED=0
MED_BENCH=True; N_MEDMCQA=1000
MMLU_CLIN=['clinical_knowledge','professional_medicine','college_medicine','anatomy','medical_genetics']
SUBSET_ORDER=['OMAQ','EHAI','FBRT-Manual','FBRT-LLM','TRINDS','CC-Manual','CC-LLM']
np.random.seed(SEED); torch.manual_seed(SEED)
def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
def memline(tag=''):
    s=''
    for d in range(torch.cuda.device_count()):
        f,t=torch.cuda.mem_get_info(d); s+=f' GPU{d} {f/1e9:.1f}/{t/1e9:.1f}'
    print(f'  [{tag}]{s} GiB free')
print('Soft-SPIN Health Step 2 | model', MODEL_ID, '| alphas', ALPHAS, '| sigmas', SIGMAS, '| MED_BENCH', MED_BENCH)

## Step 1 — coupled set + eval splits + model + capability corpora (cache originals)

In [ ]:
CP=find('coupled_soft_health.json'); assert CP, 'add coupled_soft_health.json (Soft Step-1 output) as input'
meta=json.load(open(CP)); COUPLED={k:[int(x) for x in v] for k,v in meta['coupled'].items()}
NZ=sum(len(v) for v in COUPLED.values())
print('coupled set:', CP, '| matrices', len(COUPLED), '| coupled weights', NZ)

EF=find('eval_fair_health.csv'); EP=find('eval_priv_health.csv')
assert EF and EP, 'add eval_fair_health.csv + eval_priv_health.csv (general Step-2 output) as input'
fair=pd.read_csv(EF); priv=pd.read_csv(EP)
if 'subset' not in fair.columns: fair['subset']='(all)'
def stratified(dfq, n, seed=SEED):
    subs=dfq['subset'].unique(); per=max(1,n//len(subs)); parts=[]
    for s in subs:
        d=dfq[dfq['subset']==s]; parts.append(d.sample(n=min(per,len(d)),random_state=seed))
    out=pd.concat(parts)
    return out.sample(n=n,random_state=seed) if len(out)>n else out
fair_q=stratified(fair,HSIC_N)['question'].tolist()
priv_q=priv['question'].sample(n=min(HSIC_N,len(priv)),random_state=SEED).tolist()

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
_wt=load_dataset('wikitext','wikitext-2-raw-v1',split='test')['text']
wt_ids=tokenizer('\n\n'.join(_wt), return_tensors='pt').input_ids
med_ids=None
if MED_BENCH:
    try: pq=load_dataset('pubmed_qa','pqa_labeled',split='train')
    except Exception: pq=load_dataset('pubmed_qa','pqa_labeled',split='train',trust_remote_code=True)
    cg=find('cal_general_health.csv'); used=set(pd.read_csv(cg)['prompt'].astype(str)) if cg else set()
    med_ids=tokenizer('\n\n'.join(str(a) for q,a in zip(pq['question'],pq['long_answer']) if str(q) not in used), return_tensors='pt').input_ids
print('model loaded | entry', GEN_DEV, '| fair', len(fair_q), 'priv', len(priv_q)); memline('loaded')

# cache ORIGINAL coupled-weight values ONCE (weights still untouched = baseline)
PARAMS=dict(model.named_parameters()); IDX={}; ORIG={}
for n,idxlist in COUPLED.items():
    if n in PARAMS and idxlist:
        p=PARAMS[n]; idx=torch.tensor(idxlist, device=p.device); IDX[n]=idx; ORIG[n]=p.data.view(-1)[idx].clone()
print('cached originals for', len(IDX), 'matrices,', int(sum(v.numel() for v in IDX.values())), 'weights')

## Step 2 — full-pack + capability + α-application functions

In [ ]:
def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
@torch.no_grad()
def reps_all(queries,bs=8):
    acc=None
    for i in range(0,len(queries),bs):
        enc=tokenizer(fmt(queries[i:i+bs]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        hs=model(**enc,output_hidden_states=True).hidden_states
        m=enc.attention_mask.unsqueeze(-1)
        pooled=[((h*m).sum(1)/m.sum(1)).float().cpu().numpy() for h in hs]
        acc=[[p] for p in pooled] if acc is None else [acc[l]+[pooled[l]] for l in range(len(pooled))]
    return [np.concatenate(a,0) for a in acc]
def _d2(A,B):
    sa=(A**2).sum(1,keepdims=True); sb=(B**2).sum(1,keepdims=True); return np.clip(sa+sb.T-2*A@B.T,0,None)
def hsic(X,Y,sigma):
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]
    K=np.exp(-_d2(X,X)/(2*sigma**2)); L=np.exp(-_d2(Y,Y)/(2*sigma**2)); H=np.eye(n)-np.ones((n,n))/n
    return float(np.trace(K@H@L@H)/(n-1)**2)
def nhsic(X,Y,sigma):
    hxy=hsic(X,Y,sigma); hxx=hsic(X,X,sigma); hyy=hsic(Y,Y,sigma)
    return float(hxy/np.sqrt(hxx*hyy)) if hxx>0 and hyy>0 else 0.0
def linear_cka(X,Y):
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]
    Xc=X-X.mean(0,keepdims=True); Yc=Y-Y.mean(0,keepdims=True)
    num=np.linalg.norm(Yc.T@Xc)**2; den=np.linalg.norm(Xc.T@Xc)*np.linalg.norm(Yc.T@Yc)
    return float(num/den) if den>0 else 0.0
def mean_norm(X): return float(np.linalg.norm(X,axis=1).mean())
def full_pack(Lf,Lp):
    Xf,Xp=Lf[-1],Lp[-1]
    return {'hsic_final':{str(s):hsic(Xf,Xp,s) for s in SIGMAS},
            'hsic_per_layer':{str(s):[hsic(Lf[l],Lp[l],s) for l in range(len(Lf))] for s in SIGMAS},
            'nhsic_final':{str(s):nhsic(Xf,Xp,s) for s in SIGMAS},
            'cka_final':linear_cka(Xf,Xp),
            'norm_fair':mean_norm(Xf),'norm_priv':mean_norm(Xp)}
@torch.no_grad()
def eval_ppl(ids,seqlen=PPL_SEQLEN):
    ns=ids.numel()//seqlen; nlls=[]
    for i in range(ns):
        inp=ids[:,i*seqlen:(i+1)*seqlen].to(GEN_DEV); lg=model(inp).logits
        nlls.append(torch.nn.CrossEntropyLoss()(lg[:,:-1,:].reshape(-1,lg.size(-1)),inp[:,1:].reshape(-1)).float()*seqlen)
    return torch.exp(torch.stack(nlls).sum()/(ns*seqlen)).item()
@torch.no_grad()
def _cll(prompt, choice):
    enc=tokenizer(prompt+choice, return_tensors='pt', truncation=True, max_length=1024).to(GEN_DEV)
    plen=tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
    logits=model(**enc).logits[:, :-1, :]; tgt=enc.input_ids[:, 1:]
    logp=torch.log_softmax(logits, dim=-1).gather(2, tgt.unsqueeze(-1)).squeeze(-1)[0]
    seg=logp[plen-1:]; return float(seg.mean().item()) if seg.numel()>0 else -1e9
def mc_acc(items):
    if not items: return float('nan')
    c=0
    for it in items: c+= int(np.argmax([_cll(it['prompt'],ch) for ch in it['choices']]))==it['answer']
    return c/len(items)
@torch.no_grad()
def apply_alpha(alpha):
    nz=0
    for n,idx in IDX.items():
        PARAMS[n].data.view(-1)[idx]=alpha*ORIG[n]; nz+=idx.numel()
    return int(nz)
def restore_orig(): apply_alpha(1.0)

BENCH={}
if MED_BENCH:
    try: dm=load_dataset('openlifescienceai/medmcqa',split='validation')
    except Exception: dm=load_dataset('medmcqa',split='validation',trust_remote_code=True)
    dm=dm.shuffle(seed=SEED).select(range(min(N_MEDMCQA,len(dm))))
    BENCH['MedMCQA']=[{'prompt':f"Question: {r['question']}\nAnswer:",'choices':[f" {o}" for o in (r['opa'],r['opb'],r['opc'],r['opd'])],'answer':int(r['cop'])} for r in dm]
    for s in MMLU_CLIN:
        try: ds=load_dataset('cais/mmlu',s,split='test')
        except Exception: ds=load_dataset('cais/mmlu',s,split='test',trust_remote_code=True)
        BENCH['MMLU:'+s]=[{'prompt':f"Question: {r['question']}\nAnswer:",'choices':[f" {c}" for c in r['choices']],'answer':int(r['answer'])} for r in ds]
    print('benchmarks:', {k:len(v) for k,v in BENCH.items()})
def bench_all(): return {k:mc_acc(v) for k,v in BENCH.items()} if MED_BENCH else {}
print('functions ready.')

## Step 3 — baseline once, then sweep α (checkpointed) + report table

In [ ]:
CK='/kaggle/working/metrics_soft_health_ckpt.json'
state=json.load(open(CK)) if os.path.exists(CK) else {'baseline':None,'per_alpha':{}}
REPS={}; Xf0=Xp0=None

# baseline (weights untouched = original)
if state['baseline'] is None:
    print('baseline (unmodified weights)...')
    Lf0=reps_all(fair_q); Lp0=reps_all(priv_q); base=full_pack(Lf0,Lp0)
    Xf0,Xp0=Lf0[-1],Lp0[-1]; del Lf0,Lp0; gc.collect(); torch.cuda.empty_cache()
    base['ppl']=eval_ppl(wt_ids); base['med_ppl']=eval_ppl(med_ids) if MED_BENCH else None; base['acc']=bench_all()
    state['baseline']=base; json.dump(state, open(CK,'w'))
else:
    print('baseline reused from checkpoint (bootstrap needs a single-session run — reps not cached)')
    base=state['baseline']
base_fin=base['hsic_final']
print('baseline HSIC(final):',{s:round(base_fin[str(s)],6) for s in SIGMAS},'| CKA',round(base['cka_final'],4),
      '| norms', round(base['norm_fair'],2),'/',round(base['norm_priv'],2),
      '| wiki',round(base['ppl'],4),'| pubmed',(round(base['med_ppl'],4) if MED_BENCH else 'off'))

# alpha sweep (reset to alpha*ORIG each time; absolute assignment so alphas never compound)
for a in ALPHAS:
    key=f'{a:.2f}'
    if key in state['per_alpha']:
        print(f'alpha={key}: metrics in checkpoint (skipping compute; not available for bootstrap on resume)'); continue
    nz=apply_alpha(a); t0=time.time()
    Lf=reps_all(fair_q); Lp=reps_all(priv_q); pk=full_pack(Lf,Lp); REPS[a]=(Lf[-1],Lp[-1]); del Lf,Lp; gc.collect(); torch.cuda.empty_cache()
    pk['ppl']=eval_ppl(wt_ids); pk['med_ppl']=eval_ppl(med_ids) if MED_BENCH else None; pk['acc']=bench_all()
    pk['alpha']=a; pk['coupled_scaled']=int(nz)
    state['per_alpha'][key]=pk; json.dump(state, open(CK,'w'))
    dh400=100*(pk['hsic_final']['400']-base_fin['400'])/base_fin['400']
    print(f'  alpha={key} ({time.time()-t0:.0f}s) | HSIC@400 {dh400:+.1f}% | nHSIC@400 {pk["nhsic_final"]["400"]:.4f} | CKA {pk["cka_final"]:.4f} | wiki {pk["ppl"]:.3f} | pubmed {(pk["med_ppl"] if MED_BENCH else 0):.3f}')
restore_orig(); print('weights restored to originals.')

out={'model':MODEL_ID,'domain':'healthcare','variant':'soft-SPIN','p':meta.get('p'),'q':meta.get('q'),
     'coupled_count':NZ,'sigmas':SIGMAS,'alphas':ALPHAS,'baseline':base,'per_alpha':state['per_alpha']}
json.dump(out, open('/kaggle/working/metrics_soft_health.json','w'), indent=1)
print('saved metrics_soft_health.json')

# ---- report table ----
def dr(fin,s): b=base_fin[str(s)]; return 100*(fin[str(s)]-b)/b if b else 0.0
mmc=lambda a_: (float(np.mean([a_['MMLU:'+s] for s in MMLU_CLIN])) if a_ else float('nan'))
print('\n'+'='*120)
print('  Soft-SPIN Health Step 2 — HSIC/nHSIC/CKA + capability vs alpha  (0 = hard SPIN, 1 = baseline)')
print('='*120)
hdr=f'{"alpha":>6}{"scaled":>8} '+' '.join(f'{"dH@"+str(s):>8}' for s in SIGMAS)+f'{"nH@400":>8}{"CKA":>7}{"wiki":>9}{"pubmed":>9}{"MedMCQA":>8}{"MMLUc":>7}'
print(hdr); print('-'*120)
print(f'{"base":>6}{NZ:>8} '+' '.join(f'{0.0:>+7.1f}%' for s in SIGMAS)+f'{base["nhsic_final"]["400"]:>8.4f}{base["cka_final"]:>7.4f}{base["ppl"]:>9.3f}{(base["med_ppl"] if MED_BENCH else 0):>9.3f}{base["acc"].get("MedMCQA",float("nan")):>8.3f}{mmc(base["acc"]):>7.3f}')
for a in ALPHAS:
    e=state['per_alpha'][f'{a:.2f}']
    cells_=' '.join(f'{dr(e["hsic_final"],s):>+7.1f}%' for s in SIGMAS)
    print(f'{a:>6.2f}{e["coupled_scaled"]:>8} {cells_}{e["nhsic_final"]["400"]:>8.4f}{e["cka_final"]:>7.4f}{e["ppl"]:>9.3f}{(e["med_ppl"] if MED_BENCH else 0):>9.3f}{e["acc"].get("MedMCQA",float("nan")):>8.3f}{mmc(e["acc"]):>7.3f}')
print('-'*120)
print('Expect: HSIC/nHSIC/CKA rise with alpha (0 -> baseline); perplexity lowest at alpha=1, highest at alpha=0.')

## Step 4 — bootstrap 95% CI on the HSIC drop per α (vs baseline)

In [ ]:
if BOOT and REPS and Xf0 is not None:
    n0=min(len(Xf0),len(Xp0)); rng=np.random.default_rng(0); BIDX=[rng.integers(0,n0,n0) for _ in range(BOOT)]
    def _K(X,sig): return np.exp(-_d2(X[:n0],X[:n0])/(2*sig**2))
    def _hi(K,L,idx):
        Kb=K[np.ix_(idx,idx)]; Lb=L[np.ix_(idx,idx)]; n=len(idx)
        Kc=Kb-Kb.mean(0,keepdims=True)-Kb.mean(1,keepdims=True)+Kb.mean(); return (Kc*Lb).sum()/(n-1)**2
    BK={s:(_K(Xf0,s),_K(Xp0,s)) for s in SIGMAS}; ci={}
    for a,(Xf,Xp) in REPS.items():
        for s in SIGMAS:
            Kf,Lp=_K(Xf,s),_K(Xp,s); Kf0,Lp0=BK[s]
            arr=np.array([100*((_hi(Kf,Lp,ix)-_hi(Kf0,Lp0,ix))/_hi(Kf0,Lp0,ix) if _hi(Kf0,Lp0,ix) else 0) for ix in BIDX])
            ci[f'a{a:.2f}_s{s}']={'med':float(np.median(arr)),'lo':float(np.percentile(arr,2.5)),'hi':float(np.percentile(arr,97.5))}
    out['bootstrap']={'B':BOOT,'drop_ci':ci}; json.dump(out, open('/kaggle/working/metrics_soft_health.json','w'), indent=1)
    print('HSIC drop vs baseline @ sigma=400, median [95% CI] (negative = decoupled below baseline):')
    for a in ALPHAS:
        k=f'a{a:.2f}_s400'
        if k in ci: print(f'  alpha={a:.2f}: {ci[k]["med"]:+6.1f}%  [{ci[k]["lo"]:+6.1f}, {ci[k]["hi"]:+6.1f}]')
else:
    print('bootstrap skipped (needs a single-session run with baseline + alpha reps in memory).')
print('\ndone. metrics_soft_health.json holds the full pack per alpha. Choose an operating alpha, then run Steps 3-4.')